# Longitudinal hypergraph analysis

This notebook studies how the municipality-company hypergraph changes from 2010 to 2023. Municipalities are nodes, companies are hyperedges, and an incidence means that a municipality owns a share of a company.

It is deliberately separate from influence maximization: there is no seed selection, propagation model, or connectivity constraint here. Large graph drawings are also avoided because summary statistics and distributions are more informative at this scale.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from networkx.utils import UnionFind

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## Configuration

Change `SELECTED_YEAR` for the detailed rankings. `DISTRIBUTION_YEARS` controls which years are overlaid in the distribution plots.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
SELECTED_YEAR = 2023
DISTRIBUTION_YEARS = [2010, 2015, 2020, 2023]
TOP_N = 20

year_files = {
    int(path.stem.split("_")[-1]): path
    for path in DATA_DIR.glob("rete_*.csv")
    if path.stem.split("_")[-1].isdigit()
}
YEARS = sorted(year_files)

if not YEARS:
    raise FileNotFoundError(
        "No yearly networks found. Run `python src/prepare_data.py` first."
    )
if SELECTED_YEAR not in year_files:
    raise ValueError(f"SELECTED_YEAR must be one of {YEARS}")

DISTRIBUTION_YEARS = [year for year in DISTRIBUTION_YEARS if year in year_files]
print(f"Available years: {YEARS[0]}-{YEARS[-1]} ({len(YEARS)} graphs)")

## Analysis functions

A connected component is computed on the bipartite incidence representation: municipality vertices on one side and company vertices on the other. This is only a diagnostic; disconnected graphs are retained unchanged.

In [ ]:
def load_year(year):
    frame = pd.read_csv(year_files[year])
    return frame.drop_duplicates(["CF Comune", "CF Partecipata"]).copy()


def jaccard(left, right):
    union = left | right
    return len(left & right) / len(union) if union else 1.0


def component_statistics(incidence):
    components = UnionFind()
    pairs = incidence[["CF Partecipata", "CF Comune"]].itertuples(
        index=False, name=None
    )
    for company_id, municipality_id in pairs:
        components.union(("company", company_id), ("municipality", municipality_id))

    vertices = list(components)
    sizes = Counter(components[vertex] for vertex in vertices)
    largest_share = max(sizes.values()) / len(vertices) if vertices else np.nan
    return len(sizes), largest_share


def analyze_year(year, incidence):
    edge_sizes = incidence.groupby("CF Partecipata")["CF Comune"].nunique()
    node_degrees = incidence.groupby("CF Comune")["CF Partecipata"].nunique()
    component_count, largest_component_share = component_statistics(incidence)

    summary = {
        "year": year,
        "incidences": len(incidence),
        "hyperedges": edge_sizes.size,
        "municipalities": node_degrees.size,
        "mean_edge_size": edge_sizes.mean(),
        "median_edge_size": edge_sizes.median(),
        "max_edge_size": edge_sizes.max(),
        "mean_node_degree": node_degrees.mean(),
        "median_node_degree": node_degrees.median(),
        "max_node_degree": node_degrees.max(),
        "mean_quota": incidence["Quota"].mean(),
        "median_quota": incidence["Quota"].median(),
        "total_quota": incidence["Quota"].sum(),
        "components": component_count,
        "largest_component_share": largest_component_share,
    }
    return summary, edge_sizes, node_degrees

## Build the longitudinal summary

The loop keeps only compact summaries, plus full degree distributions for the configured comparison years. Turnover is always measured between consecutive available years.

In [ ]:
annual_rows = []
turnover_rows = []
edge_size_distributions = {}
node_degree_distributions = {}
previous = None

for year in YEARS:
    incidence = load_year(year)
    summary, edge_sizes, node_degrees = analyze_year(year, incidence)
    annual_rows.append(summary)

    if year in DISTRIBUTION_YEARS:
        edge_size_distributions[year] = edge_sizes
        node_degree_distributions[year] = node_degrees

    node_set = set(incidence["CF Comune"])
    edge_set = set(incidence["CF Partecipata"])
    pair_set = set(
        incidence[["CF Comune", "CF Partecipata"]].itertuples(
            index=False, name=None
        )
    )

    if previous is not None:
        turnover_rows.append(
            {
                "year_from": previous["year"],
                "year_to": year,
                "municipality_jaccard": jaccard(previous["nodes"], node_set),
                "hyperedge_jaccard": jaccard(previous["edges"], edge_set),
                "incidence_jaccard": jaccard(previous["pairs"], pair_set),
                "new_municipalities": len(node_set - previous["nodes"]),
                "lost_municipalities": len(previous["nodes"] - node_set),
                "new_hyperedges": len(edge_set - previous["edges"]),
                "lost_hyperedges": len(previous["edges"] - edge_set),
                "new_incidences": len(pair_set - previous["pairs"]),
                "lost_incidences": len(previous["pairs"] - pair_set),
            }
        )

    previous = {"year": year, "nodes": node_set, "edges": edge_set, "pairs": pair_set}

annual_stats = pd.DataFrame(annual_rows).set_index("year")
turnover = pd.DataFrame(turnover_rows).set_index("year_to")
annual_stats

## Network evolution

Edge size is the number of municipalities in a company. Node degree is the number of companies in which a municipality participates. The largest-component share uses all municipality and company vertices in the bipartite representation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

annual_stats[["hyperedges", "municipalities"]].plot(
    ax=axes[0, 0], marker="o"
)
axes[0, 0].set_title("Active vertices")
axes[0, 0].set_ylabel("Count")

annual_stats["incidences"].plot(ax=axes[0, 1], marker="o", color="tab:purple")
axes[0, 1].set_title("Municipality-company incidences")
axes[0, 1].set_ylabel("Count")

annual_stats[["mean_edge_size", "mean_node_degree"]].plot(
    ax=axes[1, 0], marker="o"
)
axes[1, 0].set_title("Mean participation intensity")
axes[1, 0].set_ylabel("Mean count")
axes[1, 0].set_xlabel("Year")

annual_stats["largest_component_share"].plot(
    ax=axes[1, 1], marker="o", color="tab:green"
)
axes[1, 1].set_title("Largest connected component")
axes[1, 1].set_ylabel("Share of all bipartite vertices")
axes[1, 1].set_xlabel("Year")
axes[1, 1].set_ylim(0, 1.02)

fig.suptitle("Evolution of the annual hypergraphs", fontsize=16)
fig.tight_layout()
plt.show()

## Year-to-year turnover

Jaccard similarity is intersection divided by union. Values near 1 mean two consecutive years retain nearly the same objects; values near 0 mean high turnover. Incidence similarity is the strictest measure because the same municipality-company pair must persist.

In [ ]:
display(turnover)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
turnover[["municipality_jaccard", "hyperedge_jaccard", "incidence_jaccard"]].plot(
    ax=axes[0], marker="o"
)
axes[0].set_title("Similarity with the previous year")
axes[0].set_xlabel("Current year")
axes[0].set_ylabel("Jaccard similarity")
axes[0].set_ylim(0, 1.02)

turnover[["new_hyperedges", "lost_hyperedges"]].plot(
    ax=axes[1], marker="o"
)
axes[1].set_title("Hyperedge entry and exit")
axes[1].set_xlabel("Current year")
axes[1].set_ylabel("Companies")

fig.tight_layout()
plt.show()

## Degree distributions

The complementary cumulative distribution function (CCDF) shows the share of observations greater than or equal to each value. Log scales make the heavy tail visible without drawing the full graph.

In [ ]:
def plot_ccdf(series, ax, label):
    values = np.sort(np.asarray(series, dtype=float))
    survival = (len(values) - np.arange(len(values))) / len(values)
    ax.step(values, survival, where="post", label=str(label), alpha=0.85)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for year in DISTRIBUTION_YEARS:
    plot_ccdf(edge_size_distributions[year], axes[0], year)
    plot_ccdf(node_degree_distributions[year], axes[1], year)

for ax in axes:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylabel("P(X ≥ x)")
    ax.legend(title="Year")

axes[0].set_title("Hyperedge-size distribution")
axes[0].set_xlabel("Municipalities per company")
axes[1].set_title("Municipality-degree distribution")
axes[1].set_xlabel("Companies per municipality")
fig.tight_layout()
plt.show()

## Detailed view of one year

The first company ranking is purely structural. The second uses the real company-size parameter prepared from the *Partecipate* workbook; it should be interpreted together with its data-quality fields. Municipality rankings include the annual population joined through the BDAP registry.

In [ ]:
selected = load_year(SELECTED_YEAR)
parameters = pd.read_csv(DATA_DIR / "hyperedge_parameters.csv")
parameters = parameters.loc[parameters["year"].eq(SELECTED_YEAR)].copy()
node_parameters = pd.read_csv(DATA_DIR / "node_parameters.csv")
node_parameters = node_parameters.loc[
    node_parameters["year"].eq(SELECTED_YEAR)
].copy()

company_structure = (
    selected.groupby("CF Partecipata")
    .agg(
        municipalities=("CF Comune", "nunique"),
        total_quota=("Quota", "sum"),
        mean_quota=("Quota", "mean"),
        max_quota=("Quota", "max"),
    )
    .reset_index()
    .rename(columns={"CF Partecipata": "company_id"})
)

parameter_columns = [
    "company_id",
    "company_name",
    "company_size_score_0_100",
    "edge_parameter_mean_1",
    "company_size_data_confidence_0_1",
    "company_size_data_quality",
    "edge_parameter_basis",
]
company_view = company_structure.merge(
    parameters[parameter_columns], on="company_id", how="left", validate="one_to_one"
)

print(f"Top {TOP_N} hyperedges by number of municipalities in {SELECTED_YEAR}")
display(
    company_view.sort_values(
        ["municipalities", "edge_parameter_mean_1"], ascending=False
    ).head(TOP_N)
)

print(f"Top {TOP_N} hyperedges by company-size parameter in {SELECTED_YEAR}")
display(
    company_view.sort_values(
        ["edge_parameter_mean_1", "municipalities"], ascending=False
    ).head(TOP_N)
)

In [ ]:
municipality_view = (
    selected.groupby("CF Comune")
    .agg(
        municipality_name=("Comune", "first"),
        companies=("CF Partecipata", "nunique"),
        total_quota=("Quota", "sum"),
        mean_quota=("Quota", "mean"),
    )
    .reset_index()
    .rename(columns={"CF Comune": "municipality_id"})
)
municipality_view = municipality_view.merge(
    node_parameters[[
        "municipality_id",
        "municipality_bdap_id",
        "population",
        "node_weight_mean_1",
        "node_weight_basis",
    ]],
    on="municipality_id",
    how="left",
    validate="one_to_one",
)

print(f"Top {TOP_N} municipalities by hypergraph degree in {SELECTED_YEAR}")
display(
    municipality_view.sort_values(
        ["companies", "total_quota"], ascending=False
    ).head(TOP_N)
)

print(f"Top {TOP_N} municipalities by population in {SELECTED_YEAR}")
display(
    municipality_view.sort_values(
        ["node_weight_mean_1", "companies"], ascending=False
    ).head(TOP_N)
)

## Parameter data quality over time

This final diagnostic distinguishes structural change from changes in the availability of company accounting data. A neutral parameter is the within-year median fallback, not evidence that the company is truly average-sized.

In [ ]:
quality_path = DATA_DIR / "data_quality_by_year.csv"
if quality_path.exists():
    data_quality = pd.read_csv(quality_path).set_index("year")
    display(
        data_quality[[
            "company_size_available_pct",
            "neutral_parameter_fallback_pct",
            "exact_company_record_pct",
            "population_available_pct",
        ]]
    )

    ax = data_quality[[
        "company_size_available_pct",
        "neutral_parameter_fallback_pct",
        "exact_company_record_pct",
    ]].plot(figsize=(10, 4.5), marker="o")
    ax.set_title("Company-parameter data quality")
    ax.set_xlabel("Year")
    ax.set_ylabel("Percent of hyperedges")
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    print("No data-quality summary found. Run `python src/prepare_data.py`.")

## Reading the results

Useful signals to compare are: growth or contraction in active municipalities and companies; shifts in mean and tail degree; the fraction captured by the largest component; and discontinuities in year-to-year Jaccard similarity. Check low-similarity years against the data-quality plot before interpreting them as real institutional changes.